# Score the CTC head of the from-scratch model

The training log showed the CTC head decoding better than the attention decoder. This scores it with the same validation (1000) and test (3000) clips as every other model, greedy CTC decoding.

In [ ]:
BRANCH = "phase-5-scratch"
!git clone -q --branch {BRANCH} https://github.com/sohenpatel22/ASL-Fingerspelling-Recognition.git /tmp/repo
%cd /tmp/repo
!pip install -q -e .

In [ ]:
import os, zipfile, json
from pathlib import Path

def find_dir(name, root="/kaggle/input"):
    for base, dirs, _ in os.walk(root):
        if name in dirs:
            return Path(base, name)
        if base[len(root):].count(os.sep) >= 5:
            dirs[:] = []
    raise FileNotFoundError(name)

COMP = next(p for p in (Path("/kaggle/input/competitions/asl-fingerspelling"), Path("/kaggle/input/asl-fingerspelling")) if p.exists())
try:
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train"), find_dir("npy_supp")
except FileNotFoundError:
    for archive in [p for p in Path("/kaggle/input").rglob("*.zip") if "competitions" not in str(p)]:
        with zipfile.ZipFile(archive) as z:
            z.extractall("/tmp/npy")
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train", "/tmp/npy"), find_dir("npy_supp", "/tmp/npy")
DATA = f"data.comp_dir={COMP} data.npy_train={NPY_TRAIN} data.npy_supp={NPY_SUPP}"
import torch
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", "| cpus:", os.cpu_count())

In [ ]:
CKPT = next(Path("/kaggle/input").rglob("best.pth"))
print(CKPT)
!asl-eval --checkpoint {CKPT} --config configs/scratch.yaml --split val --max-samples 1000 --sample-seed 0 --decoder ctc --out /kaggle/working/val_ctc.json --details-out /kaggle/working/val_ctc_details.json {DATA}
!asl-eval --checkpoint {CKPT} --config configs/scratch.yaml --split test --max-samples 3000 --sample-seed 0 --decoder ctc --out /kaggle/working/test_ctc.json --details-out /kaggle/working/test_ctc_details.json {DATA}
val, test = (json.load(open(f"/kaggle/working/{s}_ctc.json")) for s in ("val", "test"))
print("ctc validation CER:", round(val["cer"], 4), "| test CER:", round(test["cer"], 4), "| corpus CER:", round(test["cer_micro"], 4))